In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
# The notebook will need couple module files from code directory.
import os,sys
# os.path.join('..', 'code') is the relative path for code folder, module_path convert it to absolute path, final absolute url: d:\\Study\\Python\\llm-zoomcamp\\code
module_path = os.path.abspath(os.path.join('..', 'code'))

# append the code folder in sys.path if not already exists (100% likely)
if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

#minsearch text index
documents = load_faq_data()
index = build_index(documents)

In [4]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [6]:
result=assistant.rag("How do I run Ollama locally?")
print(result)

To run Ollama locally:

1. Install Ollama from: https://ollama.com/download  
   - **macOS**: download the `.pkg` and install it  
   - **Windows**: download the `.msi` and install it  
   - **Linux**: run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. Open a terminal and start a model locally with:
```bash
ollama run llama3
```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. To test that the local server is running, you can check:
```bash
curl http://localhost:11434
```

If needed in the notebook/homework context, the Ollama server can also be restarted with:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [7]:
result=assistant.rag("How do I run Olama locally?")
print(result)

I don’t see any FAQ entry about running **Olama locally** in the provided context.

The closest related item is about running the **MCP Inspector** locally with:

```bash
npx @modelcontextprotocol/inspector
```

If you meant a different tool or have a specific setup question, please share more context.


In [8]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

print(response.output_text)

Yes—usually you can join a course after discovering it, but it depends on the course’s enrollment rules and whether registration is still open.

If you want, I can help you figure it out. Please tell me:
- the course name,
- where it’s hosted (school, platform, website, etc.),
- and whether you already have a login/account there.

If you’re asking more generally, the quickest way is to check:
1. the course page for an **Enroll / Join / Register** button,
2. the **start date** and **deadline**,
3. any **prerequisites** or approval requirements,
4. the course instructor or support team if enrollment is closed.

If you want, I can also help you write a short message asking to join.


In [9]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [10]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [11]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output)

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enroll late registration eligibility"}', call_id='call_1F5gCw29JWF4TsiAnYHYj5n6', name='search', type='function_call', id='fc_06e21cd51da74448006a2749f0ec74819d930db17c809485e8', namespace=None, status='completed')]


In [12]:
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enroll late registration eligibility"}', call_id='call_1F5gCw29JWF4TsiAnYHYj5n6', name='search', type='function_call', id='fc_06e21cd51da74448006a2749f0ec74819d930db17c809485e8', namespace=None, status='completed')

In [13]:
response.output[0].arguments

'{"query":"join course discovered course can I join enroll late registration eligibility"}'

In [14]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [18]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course

In [19]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [26]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open. If you’re just following along, you can start learning and submitting homework while the form is open, even without prior registration.'

In [27]:
usage = response.usage
usage.input_tokens, usage.output_tokens

def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [28]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [29]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [30]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course can I join discovered the course enrollment late registration"}
function_call: search {"query":"course discovered can I still join enrollment access FAQ"}
function_call: search {"query":"late enrollment join the course FAQ"}


In [32]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course can I join discovered the course enrollment late registration"}', call_id='call_yflOT4Tf8U9NwiNm7fknGwS5', name='search', type='function_call', id='fc_040da0c7e9431b8b006a274d12c25081a09e6b6a541c63c159', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course discovered can I still join enrollment access FAQ"}', call_id='call_qRLmuKPwRGZMtBOTImjR5Cue', name='search', type='function_call', id='fc_040da0c7e9431b8b006a274d12c26081a0992ba01d40202013', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"late enrollment join the course FAQ"}', call_id='call_Hf4zQvDs7ePwXC79q9LDao8w', name='search', type='function_call', id='fc_040da0c7e9431b8b006a274d12c27481a08e066ce0949e686c', namespace=None, status='completed')]

In [33]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.  

If you want a certificate, make sure you submit your project while submissions are still open. Also, if you’re aiming for a certificate, it’s only available when finishing with a live cohort, not in self-paced mode.

If you want, I can also explain how homework, registration, and the certificate process work.


In [34]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [35]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama run locally Ollama local install run locally"}
function_call: search {"query":"Ollama locally FAQ install run model local machine"}
function_call: search {"query":"run Ollama local setup FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. Install Ollama from:
   https://ollama.com/download

2. Pick your OS:
   - macOS: install the `.pkg`
   - Windows: install the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

3. Start a model locally:
   ```bash
   ollama run llama3
   ```

   This will download the model and open a local chat interface.

4. If you want to check that the local server is running:
   ```bash
   curl http://localhost:11434
   ```

5. If you want to call Ollama from Python:
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )



'To run Ollama locally:\n\n1. Install Ollama from:\n   https://ollama.com/download\n\n2. Pick your OS:\n   - macOS: install the `.pkg`\n   - Windows: install the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n3. Start a model locally:\n   ```bash\n   ollama run llama3\n   ```\n\n   This will download the model and open a local chat interface.\n\n4. If you want to check that the local server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n\n5. If you want to call Ollama from Python:\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection issue in a notebook or homework setup, restarting the server can help:\n```bash\nnohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you want, I can also s

In [36]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment can I still join discovered the course"}
function_call: search {"query":"enrollment late join course FAQ discovered course"}
function_call: search {"query":"can I still join the course after it started FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. Also, if you’re aiming for the certificate, note that it’s only available for the live cohort, not self-paced study.

If you want, I can also help with questions about homework, certificates, or how to start catching up.


'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open. Also, if you’re aiming for the certificate, note that it’s only available for the live cohort, not self-paced study.\n\nIf you want, I can also help with questions about homework, certificates, or how to start catching up.'

In [37]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join the course late enroll discovered course can I join FAQ"}
iteration #2...
function_call: search {"query":"certificate project while accepting submissions self-paced peer-review live cohort FAQ project submission still accepting submissions"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. After that, self-paced participation won’t qualify for a certificate because the course requires live cohort peer review.

If you’d like, I can also explain how the certificate/peer-review process works. Any other areas you want to explore?


'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open. After that, self-paced participation won’t qualify for a certificate because the course requires live cohort peer review.\n\nIf you’d like, I can also explain how the certificate/peer-review process works. Any other areas you want to explore?'

In [38]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit definition course FAQ"}
function_call: search {"query":"what is the queen gambit chess"}
function_call: search {"query":"queen gambit opening FAQ"}
iteration #2...
ASSISTANT:
If you mean the chess opening, the **Queen’s Gambit** is an opening that starts with:

1. d4 d5  
2. c4

White offers the c-pawn to try to gain control of the center and distract Black’s d-pawn. It’s one of the most famous and classical chess openings.

If you meant something else by “queen gambit,” tell me the context and I’ll explain that too. Want a quick overview of the main ideas and typical lines?


'If you mean the chess opening, the **Queen’s Gambit** is an opening that starts with:\n\n1. d4 d5  \n2. c4\n\nWhite offers the c-pawn to try to gain control of the center and distract Black’s d-pawn. It’s one of the most famous and classical chess openings.\n\nIf you meant something else by “queen gambit,” tell me the context and I’ll explain that too. Want a quick overview of the main ideas and typical lines?'

In [39]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen’s gambit,” so it looks like this is off-topic or not covered in the course materials I have access to.

If you meant something else related to the course, feel free to rephrase your question. Are there other areas you want to explore?


'I couldn’t find a course FAQ entry for “queen’s gambit,” so it looks like this is off-topic or not covered in the course materials I have access to.\n\nIf you meant something else related to the course, feel free to rephrase your question. Are there other areas you want to explore?'

In [40]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [41]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [42]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [43]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [44]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [45]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [46]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [47]:
result.cost

CostInfo(input_cost=Decimal('0.002658'), output_cost=Decimal('0.0014355'), total_cost=Decimal('0.0040935'))

In [48]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local install Ollama"}', call_id='call_KSlex

In [49]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [50]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='how to improve my english speaking skills?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"improve english speaking 